# Plots

Plots for the jensenRummel2026 paper


Instructions: uses the conda environment via
```bash
cd PATH/TO/Spheroidal3D-collisions/Janus_3D_code/LCPsolvers/jensenRummel2025
conda env create -f environment.yml
```
If you need to update the conda environment file use:
```bash
conda env export --no-builds > environment.yml
```

In [1]:
# Includes
%load_ext autoreload
%autoreload 2
import os, warnings, bson
import numpy as np
from datetime import datetime
# Plotlyjs
import plotly.graph_objects as go
# matplotlib
# %matplotlib inline
# import matplotlib.pyplot as plt
# import matplotlib.ticker as ticker
# from matplotlib.ticker import AutoMinorLocator
# File IO -> mat files
import scipy.io

In [2]:
BASE_DIR = "/Users/niru8088/scratch/Spheroidal3D-collisions"
FIG_DIR = "/Users/niru8088/scratch/Spheroidal3D-collisions/docs/fig"

In [3]:
# colors : Cool Gray, Steel Blue, Deep Plum, Slate Blue, Terracotta, 
#          Seafoam Green, Burnt Orange, Mustard Gold Deep Indigo
solid_colors = ["#006450",  "#4664AA", "#78828C", "#2D4178", "#A06428", "#643C73", # 7C - 8b+
                "#DAA520",  "#287882", "#B4503C"] # 8c - 9a
        # ,]
shade_colors_hex = [ "#00645033","#4664AA33","#78828C33", "#2D417833", "#A0642833", "#643C7333", # 7C - 8b+
                 "#DAA52040",  "#28788233", "#B4503C33"] # 8c - 9a 
                # ]
shade_colors = ["rgba(0, 100, 80, 0.2)","rgba(70, 100, 170, 0.2)", "rgba(120, 130, 140, 0.2)","rgba(45, 65, 120, 0.2)","rgba(160, 100, 40, 0.2)", "rgba(100, 60, 115, 0.2)", # 7c - 8b+
                "rgba(218, 165, 32, 0.25)","rgba(40, 120, 130, 0.2)","rgba(180, 80, 60, 0.2)"] # 8c - 9a 
                # ,]
# Neutrals :  Carbon Black, Warm Slate
neutral_solid = ["rgb(40, 40, 40)", "rgb(100, 95, 90)",]
neutral_shade = ["rgba(40, 40, 40, 0.15)","rgba(100, 95, 90, 0.2)"]

# Time Analysis

In [39]:
from datetime import timedelta
# 
def getFullRunTime(iterTimes, timeData, ixEnd=None):
    Nt = len(iterTimes)
    if ixEnd is None: 
        ixEnd = np.where(iterTimes > 0)[0][-1]
    if ixEnd != Nt:
        estimatedTimeSeconds = (timeData['timings']['setup_kernel'][0][0][0] + timeData['timings']['setup_surf'][0][0][0] 
            + np.sum(iterTimes[:ixEnd]) + iterTimes[ixEnd] * (Nt - ixEnd))
        estimatedTimeSeconds = estimatedTimeSeconds[0]
        return estimatedTimeSeconds, str(timedelta(seconds=estimatedTimeSeconds))
    # If the run finished don't estimate anything
    timeSeconds = timeData['timings']['fullRunTime'][0][0][0]
    timeSeconds = timeSeconds[0]
    return timeSeconds, str(timedelta(seconds=timeSeconds))


def getCollisionRunTime(colTimesPerIter, ixEnd=None):
    Nt = len(colTimesPerIter)
    if ixEnd is None: 
        ixEnd = np.where(colTimesPerIter > 0)[0][-1]
    if ixEnd != Nt:
        estimatedTimeSeconds = np.sum(colTimesPerIter[:ixEnd]) + colTimesPerIter[ixEnd] * (Nt - ixEnd)
        return estimatedTimeSeconds, str(timedelta(seconds=estimatedTimeSeconds))
    # If the run finished don't estimate anything
    timeSeconds = np.sum(colTimesPerIter)
    return timeSeconds, str(timedelta(seconds=timeSeconds))

## n = 3

In [5]:
bbpgd_timeData = scipy.io.loadmat(
    os.path.join(
        "/Users/niru8088/scratch/Spheroidal3D-collisions/Janus_3D_code/resultsForRecord",
        'amphi.lattice.n_3.p_8.cDist_3.lcpSlvr_bbpgd.polyDisperseRatio_0.2verification.pLo_4.profile.mat'
    )
)
pqn_timeData = scipy.io.loadmat(
    os.path.join(
        "/Users/niru8088/scratch/Spheroidal3D-collisions/Janus_3D_code/resultsForRecord/",
        "amphi.lattice.n_3.p_8.cDist_3.lcpSlvr_proxquasinewton.polyDisperseRatio_0.2verification.pLo_4.profile.mat"
    )
)
bpqn_timeData = scipy.io.loadmat(
    os.path.join(
        "/Users/niru8088/scratch/Spheroidal3D-collisions/Janus_3D_code/resultsForRecord",
        'amphi.lattice.n_3.p_8.cDist_3.lcpSlvr_bifi.polyDisperseRatio_0.2_noWarmStart.profile.mat'
    )
)

In [6]:
# Create a matrix where each column is the total time for each iteration of each algorithm
matrix = np.hstack((bbpgd_timeData['timings']['total'][0][0], pqn_timeData['timings']['total'][0][0], bpqn_timeData['timings']['total'][0][0]))
cMatrix = np.hstack((bbpgd_timeData['timings']['velocities'][0][0]['col'][0][0], pqn_timeData['timings']['velocities'][0][0]['col'][0][0], bpqn_timeData['timings']['velocities'][0][0]['col'][0][0]))
algoNames = ['BB-PGD', 'PQN', 'B-PQN']
iters = np.arange(start=1, stop=matrix.shape[0]+1)

In [7]:
## Plot the timing info per iteration for each algorithm 
fig = go.Figure()
## Loop through the columns of the data matrix and make a line for each algorithm 
for i, name in zip(range(matrix.shape[1]), algoNames):
    fig.add_trace(go.Scatter(
        x=iters,
        y=matrix[:, i],
        mode='lines',
        name="total",
        legendgroup=name,
        legendgrouptitle=dict(text=name),
        line_color=solid_colors[i],
    ))
    fig.add_trace(go.Scatter(
        x=iters,
        y=cMatrix[:, i],
        mode='lines',
        name="collision",
        legendgroup=name,
        line_color=solid_colors[i],
        line_dash="dash"
    ))

## Update the Layout
fig.update_layout(
    title=dict(
        x=0.5,
        text="<b>Time Per Iteration</b>",
        font=dict(size=22) 
    ),
    xaxis=dict(
        title="Iteration", 
        # tickvals=np.arange(start=0, stop=len(labels)),
        # ticktext=[f'{yr}<br>{name}' for name, yr in zip(labels,climberBestYears)],
        # range=[-0.6, len(labels)-.4],
        # zeroline=False,
        showgrid=False,
        tickfont=dict(size=14),   
        title_font=dict(size=18)
    ),
    yaxis=dict(
        title='<b>Time (s)</b>',
        # tickvals=bin_centers,
        # ticktext=[SCORE_TO_FONT.get(int(t), t) for t in bin_centers],
        gridcolor='lightgrey',
        tickfont=dict(size=14),   
        title_font=dict(size=18)
    ),
    legend=dict(
        title=dict(
            text="Total Time Per Iteration",
            font_size=18
        ),
        orientation="v",      # Vertical orientation is standard for side legends
        yanchor="middle",     # Centers the legend vertically relative to the 'y' value
        y=0.5,                # Places legend at the vertical midpoint (50%)
        xanchor="left",       # Anchors the left side of the legend to the 'x' coordinate
        x=1.02,               # Moves it slightly to the right of the plot area
        bgcolor='rgba(255, 255, 255, 0.5)', # Optional: semi-transparent background
        bordercolor="Black",
        borderwidth=1
    ),
    font=dict(
        family="Arial, sans-serif", # Clean, web-safe font
        size=15,                   # Global default size
        color="black"
    ),
    hovermode='x unified',
    barmode='overlay', # Crucial for overlapping All/FA/Flash
    plot_bgcolor='white',
    # Increase the right margin so the legend doesn't get cut off
    margin=dict(r=150),
    height=600,
    autosize=True
)

config = {'responsive': True}
fig.show(config=config)

## Save to pdf file
# fig.write_image("totalTiming.pdf", width=800, height=600, scale=2)

## Save the figure as a standalone interactive HTML file
# html_file = '.html'
# height_script = """
#     var sendHeight = function() {
#         var height = document.body.scrollHeight;
#         window.parent.postMessage({ 'height': height }, "*");
#     };
#     window.onload = sendHeight;
#     // Also send height if the window is resized
#     window.onresize = sendHeight;
# """
# fig.write_html(os.path.join(FIG_DIR, html_file), full_html=True, include_plotlyjs='cdn', config=config, post_script=height_script)

In [31]:
print(f'Total Run Time')
# bbpgd
bbpgd_seconds,bbpgd_duration = getFullRunTime(matrix[:,0], bbpgd_timeData)
# pqn
pqn_seconds,pqn_duration =  getFullRunTime(matrix[:,1], pqn_timeData)
# bifi
bpqn_seconds,bpqn_duration =  getFullRunTime(matrix[:,2], bpqn_timeData)
print(f'- Durations')
print(f'-- {bbpgd_duration=}')
print(f'-- {pqn_duration=}')
print(f'-- {bpqn_duration=}')
#
pqn_speedUp = bbpgd_seconds / pqn_seconds
print(f'- PQN is {pqn_speedUp:.2f} faster than PGD')
#
bpqn_speedUp = bbpgd_seconds / bpqn_seconds
print(f'- B-PQN is {bpqn_speedUp:.2f} faster than PGD')

Total Run Time
- Durations
-- bbpgd_duration='7 days, 13:21:12.365149'
-- pqn_duration='6 days, 16:08:22.259998'
-- bpqn_duration='5 days, 8:38:54.842918'
- PQN is 1.13 faster than PGD
- B-PQN is 1.41 faster than PGD


In [9]:
print(f'Collision Resolution Run Time')
# bbpgd
bbpgd_seconds,bbpgd_duration = getCollisionRunTime(cMatrix[:,0],ixEnd=57)
# pqn
pqn_seconds,pqn_duration =  getCollisionRunTime(cMatrix[:,1], ixEnd=57)
# bifi
bpqn_seconds,bpqn_duration =  getCollisionRunTime(cMatrix[:,2], ixEnd=57)
print(f'- Durations')
print(f'-- {bbpgd_duration=}')
print(f'-- {pqn_duration=}')
print(f'-- {bpqn_duration=}')
#
pqn_speedUp = bbpgd_seconds / pqn_seconds
print(f'- PQN is {pqn_speedUp:.2f} faster than PGD')
#
bpqn_speedUp = bbpgd_seconds / bpqn_seconds
print(f'- B-PQN is {bpqn_speedUp:.2f} faster than PGD')

Collision Resolution Run Time
- Durations
-- bbpgd_duration='12:39:42.396773'
-- pqn_duration='10:47:55.469876'
-- bpqn_duration='7:30:36.546518'
- PQN is 1.17 faster than PGD
- B-PQN is 1.69 faster than PGD


## n = 5 

In [73]:
RESULTS_DIR = '/Users/niru8088/scratch/Spheroidal3D-collisions/Janus_3D_code/resultsForRecord'
bbpgd_timeData = scipy.io.loadmat(
    os.path.join(
        RESULTS_DIR,
        'amphi.lattice.n_5.p_8.cDist_3.lcpSlvr_bbpgd.polyDisperseRatio_0.2verification.new.profile.mat'
    )
)
pqn_timeData = scipy.io.loadmat(
    os.path.join(
        RESULTS_DIR,
        'amphi.lattice.n_5.p_8.cDist_3.lcpSlvr_proxquasinewton.polyDisperseRatio_0.2verification.new.profile.mat'
    )
)
bpqn_timeData = scipy.io.loadmat(
    os.path.join(
        RESULTS_DIR,
        'amphi.lattice.n_5.p_8.cDist_3.lcpSlvr_bifi.polyDisperseRatio_0.2verification.pLo_4.profile.mat'
    )
)

bpqn_noWarmStart_timeData = scipy.io.loadmat(
    os.path.join(
        RESULTS_DIR,
        'amphi.lattice.n_5.p_8.cDist_3.lcpSlvr_bifi.polyDisperseRatio_0.2_noWarmStart.profile.mat'
    )
)

In [74]:
# Create a matrix where each column is the total time for each iteration of each algorithm
matrix = np.hstack((bbpgd_timeData['timings']['total'][0][0], pqn_timeData['timings']['total'][0][0], bpqn_timeData['timings']['total'][0][0], bpqn_noWarmStart_timeData['timings']['total'][0][0]))
cMatrix = np.hstack((bbpgd_timeData['timings']['velocities'][0][0]['col'][0][0], pqn_timeData['timings']['velocities'][0][0]['col'][0][0], bpqn_timeData['timings']['velocities'][0][0]['col'][0][0], bpqn_noWarmStart_timeData['timings']['velocities'][0][0]['col'][0][0]))
algoNames = ['BB-PGD', 'PQN', 'B-PQN', 'B-PQN (No Warm Start)']
iters = np.arange(start=1, stop=matrix.shape[0]+1)

In [75]:
## Plot the timing info per iteration for each algorithm 
fig = go.Figure()
## Loop through the columns of the data matrix and make a line for each algorithm 
for i, name in zip(range(matrix.shape[1]), algoNames):
    fig.add_trace(go.Scatter(
        x=iters,
        y=matrix[:, i],
        mode='lines',
        name="total",
        legendgroup=name,
        legendgrouptitle=dict(text=name),
        line_color=solid_colors[i],
    ))
    fig.add_trace(go.Scatter(
        x=iters,
        y=cMatrix[:, i],
        mode='lines',
        name="collision",
        legendgroup=name,
        line_color=solid_colors[i],
        line_dash="dash"
    ))

## Update the Layout
fig.update_layout(
    title=dict(
        x=0.5,
        text="$n=5$",#"<b>"+r"$n=5$"+"</b>\n<b>Time Per Iteration</b>",
        font=dict(size=22) 
    ),
    xaxis=dict(
        title="Iteration", 
        # tickvals=np.arange(start=0, stop=len(labels)),
        # ticktext=[f'{yr}<br>{name}' for name, yr in zip(labels,climberBestYears)],
        # range=[-0.6, len(labels)-.4],
        # zeroline=False,
        showgrid=False,
        tickfont=dict(size=14),   
        title_font=dict(size=18)
    ),
    yaxis=dict(
        title='<b>Time (s)</b>',
        # tickvals=bin_centers,
        # ticktext=[SCORE_TO_FONT.get(int(t), t) for t in bin_centers],
        gridcolor='lightgrey',
        tickfont=dict(size=14),   
        title_font=dict(size=18)
    ),
    legend=dict(
        title=dict(
            text="Total Time Per Iteration",
            font_size=18
        ),
        orientation="v",      # Vertical orientation is standard for side legends
        yanchor="middle",     # Centers the legend vertically relative to the 'y' value
        y=0.5,                # Places legend at the vertical midpoint (50%)
        xanchor="left",       # Anchors the left side of the legend to the 'x' coordinate
        x=1.02,               # Moves it slightly to the right of the plot area
        bgcolor='rgba(255, 255, 255, 0.5)', # Optional: semi-transparent background
        bordercolor="Black",
        borderwidth=1
    ),
    font=dict(
        family="Arial, sans-serif", # Clean, web-safe font
        size=15,                   # Global default size
        color="black"
    ),
    hovermode='x unified',
    barmode='overlay', # Crucial for overlapping All/FA/Flash
    plot_bgcolor='white',
    # Increase the right margin so the legend doesn't get cut off
    margin=dict(r=150),
    height=600,
    autosize=True
)

config = {'responsive': True}
fig.show(config=config)

## Save to pdf file
# fig.write_image("totalTiming.pdf", width=800, height=600, scale=2)

## Save the figure as a standalone interactive HTML file
# html_file = '.html'
# height_script = """
#     var sendHeight = function() {
#         var height = document.body.scrollHeight;
#         window.parent.postMessage({ 'height': height }, "*");
#     };
#     window.onload = sendHeight;
#     // Also send height if the window is resized
#     window.onresize = sendHeight;
# """
# fig.write_html(os.path.join(FIG_DIR, html_file), full_html=True, include_plotlyjs='cdn', config=config, post_script=height_script)

In [76]:
print(f'Total Run Time')
# bbpgd
bbpgd_seconds,bbpgd_duration = getFullRunTime(matrix[2:,0], bbpgd_timeData)
# pqn
pqn_seconds,pqn_duration =  getFullRunTime(matrix[2:,1], pqn_timeData)
# bifi
bpqn_seconds,bpqn_duration =  getFullRunTime(matrix[2:,2], bpqn_timeData)
# bifi no warm start
bpqn_nws_seconds,bpqn_nws_duration =  getFullRunTime(matrix[2:,3], bpqn_timeData, ixEnd=112)
print(f'- Durations')
print(f'-- {bbpgd_duration=}')
print(f'-- {pqn_duration=}')
print(f'-- {bpqn_duration=}')
print(f'-- {bpqn_nws_duration=}')
#
pqn_speedUp = bbpgd_seconds / pqn_seconds
print(f'- PQN is {pqn_speedUp:.2f} faster than PGD')
#
bpqn_speedUp = bbpgd_seconds / bpqn_seconds
print(f'- B-PQN is {bpqn_speedUp:.2f} faster than PGD')
#
bpqn_nws_speedUp = bbpgd_seconds / bpqn_nws_seconds
print(f'- B-PQN (no warm start) is {bpqn_nws_speedUp:.2f} faster than PGD')

Total Run Time
- Durations
-- bbpgd_duration='2 days, 17:33:16.111432'
-- pqn_duration='2 days, 1:43:53.933560'
-- bpqn_duration='1 day, 11:18:48.998292'
-- bpqn_nws_duration='1 day, 23:30:08.782909'
- PQN is 1.32 faster than PGD
- B-PQN is 1.86 faster than PGD
- B-PQN (no warm start) is 1.38 faster than PGD


In [77]:
print(f'Collision Resolution Run Time')
# bbpgd
bbpgd_seconds, bbpgd_duration = getCollisionRunTime(cMatrix[2:,0])
# pqn
pqn_seconds, pqn_duration =  getCollisionRunTime(cMatrix[2:,1])
# bifi
bpqn_seconds, bpqn_duration =  getCollisionRunTime(cMatrix[2:,2])
# bifi (no warm start)
bpqn_nws_seconds, bpqn_nws_duration =  getCollisionRunTime(cMatrix[2:,3], ixEnd=112)
print(f'- Durations')
print(f'-- {bbpgd_duration=}')
print(f'-- {pqn_duration=}')
print(f'-- {bpqn_duration=}')
print(f'-- {bpqn_nws_duration=}')
#
pqn_speedUp = bbpgd_seconds / pqn_seconds
print(f'- PQN is {pqn_speedUp:.2f} faster than PGD')
#
bpqn_speedUp = bbpgd_seconds / bpqn_seconds
print(f'- B-PQN is {bpqn_speedUp:.2f} faster than PGD')
#
bpqn_nws_speedUp = bbpgd_seconds / bpqn_nws_seconds
print(f'- B-PQN (no warm start) is {bpqn_nws_speedUp:.2f} faster than PGD')

Collision Resolution Run Time
- Durations
-- bbpgd_duration='1 day, 23:59:55.656770'
-- pqn_duration='1 day, 9:37:25.303593'
-- bpqn_duration='20:10:14.765433'
-- bpqn_nws_duration='1 day, 7:44:29.874428'
- PQN is 1.43 faster than PGD
- B-PQN is 2.38 faster than PGD
- B-PQN (no warm start) is 1.51 faster than PGD


## n = 6

In [78]:
RESULTS_DIR = '/Users/niru8088/scratch/Spheroidal3D-collisions/Janus_3D_code/resultsForRecord'
bbpgd_timeData = scipy.io.loadmat(
    os.path.join(
        RESULTS_DIR,
        "amphi.lattice.n_6.p_8.cDist_3.lcpSlvr_bbpgd.polyDisperseRatio_0.2verification.new.profile.mat"
    )
)
pqn_timeData = scipy.io.loadmat(
    os.path.join(
        RESULTS_DIR,
        "amphi.lattice.n_6.p_8.cDist_3.lcpSlvr_proxquasinewton.polyDisperseRatio_0.2verification.new.profile.mat"
    )
)
bpqn_timeData = scipy.io.loadmat(
    os.path.join(
        RESULTS_DIR,
        'amphi.lattice.n_6.p_8.cDist_3.lcpSlvr_bifi.polyDisperseRatio_0.2verification.pLo_4.profile.mat'
    )
)
bpqn_noWarmStart_timeData = scipy.io.loadmat(
    os.path.join(
        RESULTS_DIR,
        'amphi.lattice.n_6.p_8.cDist_3.lcpSlvr_bifi.polyDisperseRatio_0.2_noWarmStart.profile.mat'
    )
)

In [79]:
# Create a matrix where each column is the total time for each iteration of each algorithm
matrix = np.hstack((
    bbpgd_timeData['timings']['total'][0][0], 
    pqn_timeData['timings']['total'][0][0], 
    bpqn_timeData['timings']['total'][0][0], 
    bpqn_noWarmStart_timeData['timings']['total'][0][0]
))
cMatrix = np.hstack((
    bbpgd_timeData['timings']['velocities'][0][0]['col'][0][0], 
    pqn_timeData['timings']['velocities'][0][0]['col'][0][0], 
    bpqn_timeData['timings']['velocities'][0][0]['col'][0][0], 
    bpqn_noWarmStart_timeData['timings']['velocities'][0][0]['col'][0][0]
))
algoNames = [
    'BB-PGD', 
    'PQN', 
    'B-PQN', 
    'B-PQN (No Warm Start)'
]
iters = np.arange(start=1, stop=matrix.shape[0]+1)

In [80]:
## Plot the timing info per iteration for each algorithm 
fig = go.Figure()
## Loop through the columns of the data matrix and make a line for each algorithm 
for i, name in zip(range(matrix.shape[1]), algoNames):
    fig.add_trace(go.Scatter(
        x=iters,
        y=matrix[:, i],
        mode='lines',
        name="total",
        legendgroup=name,
        legendgrouptitle=dict(text=name),
        line_color=solid_colors[i],
    ))
    fig.add_trace(go.Scatter(
        x=iters,
        y=cMatrix[:, i],
        mode='lines',
        name="collision",
        legendgroup=name,
        line_color=solid_colors[i],
        line_dash="dash"
    ))

## Update the Layout
fig.update_layout(
    title=dict(
        x=0.5,
        text="<b>Time Per Iteration</b>",
        font=dict(size=22) 
    ),
    xaxis=dict(
        title="Iteration", 
        # tickvals=np.arange(start=0, stop=len(labels)),
        # ticktext=[f'{yr}<br>{name}' for name, yr in zip(labels,climberBestYears)],
        # range=[-0.6, len(labels)-.4],
        # zeroline=False,
        showgrid=False,
        tickfont=dict(size=14),   
        title_font=dict(size=18)
    ),
    yaxis=dict(
        title='<b>Time (s)</b>',
        # tickvals=bin_centers,
        # ticktext=[SCORE_TO_FONT.get(int(t), t) for t in bin_centers],
        gridcolor='lightgrey',
        tickfont=dict(size=14),   
        title_font=dict(size=18)
    ),
    legend=dict(
        title=dict(
            text="Total Time Per Iteration",
            font_size=18
        ),
        orientation="v",      # Vertical orientation is standard for side legends
        yanchor="middle",     # Centers the legend vertically relative to the 'y' value
        y=0.5,                # Places legend at the vertical midpoint (50%)
        xanchor="left",       # Anchors the left side of the legend to the 'x' coordinate
        x=1.02,               # Moves it slightly to the right of the plot area
        bgcolor='rgba(255, 255, 255, 0.5)', # Optional: semi-transparent background
        bordercolor="Black",
        borderwidth=1
    ),
    font=dict(
        family="Arial, sans-serif", # Clean, web-safe font
        size=15,                   # Global default size
        color="black"
    ),
    hovermode='x unified',
    barmode='overlay', # Crucial for overlapping All/FA/Flash
    plot_bgcolor='white',
    # Increase the right margin so the legend doesn't get cut off
    margin=dict(r=150),
    height=600,
    autosize=True
)

config = {'responsive': True}
fig.show(config=config)

## Save to pdf file
# fig.write_image("totalTiming.pdf", width=800, height=600, scale=2)

## Save the figure as a standalone interactive HTML file
# html_file = '.html'
# height_script = """
#     var sendHeight = function() {
#         var height = document.body.scrollHeight;
#         window.parent.postMessage({ 'height': height }, "*");
#     };
#     window.onload = sendHeight;
#     // Also send height if the window is resized
#     window.onresize = sendHeight;
# """
# fig.write_html(os.path.join(FIG_DIR, html_file), full_html=True, include_plotlyjs='cdn', config=config, post_script=height_script)

In [81]:
print(f'Total Run Time')
# bbpgd
bbpgd_seconds,bbpgd_duration = getFullRunTime(matrix[:,0], bbpgd_timeData)
# pqn
pqn_seconds,pqn_duration =  getFullRunTime(matrix[:,1], pqn_timeData)
# bifi
bpqn_seconds,bpqn_duration =  getFullRunTime(matrix[:,2], bpqn_timeData)
# bifi no warm start
bpqn_nws_seconds,bpqn_nws_duration =  getFullRunTime(matrix[:,3], bpqn_timeData)
print(f'- Durations')
print(f'-- {bbpgd_duration=}')
print(f'-- {pqn_duration=}')
print(f'-- {bpqn_duration=}')
print(f'-- {bpqn_nws_duration=}')
#
pqn_speedUp = bbpgd_seconds / pqn_seconds
print(f'- PQN is {pqn_speedUp:.2f} faster than PGD')
#
bpqn_speedUp = bbpgd_seconds / bpqn_seconds
print(f'- B-PQN is {bpqn_speedUp:.2f} faster than PGD')
#
bpqn_nws_speedUp = bbpgd_seconds / bpqn_nws_seconds
print(f'- B-PQN (no warm start) is {bpqn_nws_speedUp:.2f} faster than PGD')

Total Run Time
- Durations
-- bbpgd_duration='7 days, 18:00:28.993418'
-- pqn_duration='6 days, 16:16:19.079939'
-- bpqn_duration='5 days, 13:42:39.741464'
-- bpqn_nws_duration='3 days, 9:43:37.864937'
- PQN is 1.16 faster than PGD
- B-PQN is 1.39 faster than PGD
- B-PQN (no warm start) is 2.28 faster than PGD


In [72]:
print(f'Collision Resolution Run Time')
# bbpgd
bbpgd_seconds, bbpgd_duration = getCollisionRunTime(cMatrix[100:183,0])
# pqn
pqn_seconds, pqn_duration =  getCollisionRunTime(cMatrix[100:183,1])
# bifi
bpqn_seconds, bpqn_duration =  getCollisionRunTime(cMatrix[100:183,2])
# bifi (no warm start)
# bpqn_nws_seconds, bpqn_nws_duration =  getCollisionRunTime(cMatrix[100:183,3])
print(f'- Durations')
print(f'-- {bbpgd_duration=}')
print(f'-- {pqn_duration=}')
print(f'-- {bpqn_duration=}')
# print(f'-- {bpqn_nws_duration=}')
#
pqn_speedUp = bbpgd_seconds / pqn_seconds
print(f'- PQN is {pqn_speedUp:.2f} faster than PGD')
#
bpqn_speedUp = bbpgd_seconds / bpqn_seconds
print(f'- B-PQN is {bpqn_speedUp:.2f} faster than PGD')
#
# bpqn_nws_speedUp = bbpgd_seconds / bpqn_nws_seconds
# print(f'- B-PQN (no warm start) is {bpqn_nws_speedUp:.2f} faster than PGD')

Collision Resolution Run Time
- Durations
-- bbpgd_duration='2 days, 14:10:01.417472'
-- pqn_duration='2 days, 0:04:59.242203'
-- bpqn_duration='1 day, 5:33:15.376300'
- PQN is 1.29 faster than PGD
- B-PQN is 2.10 faster than PGD


## Log File Scraping

In [26]:
def scrape_log(fn):
    with open(os.path.join(RESULTS_DIR, fn), 'r') as file:
        # .readlines() creates a list where each element is one line from the file
        lines = file.readlines()

    # Stores tuples of (line_number, line_content)
    indices = [(i,line) for i, line in enumerate(lines) if 'Total computing time' in line]
    startIx = 0
    lines_per_timestep = []
    for i, content in indices:
        # print(f"Line {i}: {content.rstrip()}")
        # print(lines[i])
        lines_per_timestep.append(lines[startIx:i])
        startIx = i
    ## Collect summary stats 
    lcp_sizes = []
    mvp_times = []
    lofi_mvp_times = []
    mvp_cnts = []
    lofi_mvp_cnts = []
    lcp_solve_times = []
    lcp_solve_tol = []

    for lines in lines_per_timestep:
        ## Calculate the size of the lcp
        ix_and_line_before_sz = [(i,line) for i, line in enumerate(lines) if 'Time for contact force correction' in line][0]
        lines_with_cols = [line for line in lines[:ix_and_line_before_sz[0]] if 'Columns' in line]
        n = int(lines_with_cols[-1].split(' ')[-1])
        print(f'- {n=}')
        ## Compute mvp info
        lines_with_MVP = [(i,line) for i, line in enumerate(lines) if 'A[x] MVP time' in line]
        lines_with_lofi_MVP = [(i,line) for i, line in enumerate(lines) if 'Ahat[x] MVP time' in line]
        _mvp_times = []
        for i, line in lines_with_MVP:
            # print(f'- {line=}')
            parts = line.split('time ')
            parts = parts[-1].split(' sec')
            _mvp_times.append(float(parts[0].strip()))
        print(f'- {_mvp_times=}')
        _lofi_mvp_times = []
        for i, line in lines_with_lofi_MVP:
            # print(f'- {line=}')
            parts = line.split('time ')
            parts = parts[-1].split(' sec')
            _lofi_mvp_times.append(float(parts[0].strip()))
        print(f"- {_lofi_mvp_times=}")
        num_mvps = len(lines_with_MVP)
        num_lofi_mvps = len(lines_with_lofi_MVP)
        ix_and_line_with_LCP_sol = [(i,line) for i, line in enumerate(lines) if 'LCP solution error' in line][0]

        parts = ix_and_line_with_LCP_sol[1].split('=')
        lcp_error = float(parts[1].split(',')[0].strip())
        lcp_iters = int(parts[2].strip())
        line_with_solve_time = lines[ix_and_line_with_LCP_sol[0]+2]
        parts = line_with_solve_time.split('=')
        lcp_solve_time = float(parts[1].strip())
        print(f'- {lcp_solve_time=}')

        lcp_sizes.append(n)
        mvp_times.append(_mvp_times)
        lofi_mvp_times.append(_lofi_mvp_times)
        mvp_cnts.append(num_mvps)
        lofi_mvp_cnts.append(num_lofi_mvps)
        lcp_solve_times.append(lcp_solve_time)
        lcp_solve_tol.append(lcp_error)
    return {
        'lcp_sizes': lcp_sizes,
        'mvp_times': mvp_times,
        'lofi_mvp_times': lofi_mvp_times,
        'mvp_cnts': mvp_cnts,
        'lofi_mvp_cnts': lofi_mvp_cnts,
        'lcp_solve_times': lcp_solve_times,
        'lcp_solve_tol': lcp_solve_tol,
    }


algo_names = [
    'BB-PGD',
    'Monofidelity PQN',
    'B-PQN',
    'B-PQN (No Warm Start)'
]
file_names =[
    'amphi.lattice.n_6.p_8.cDist_3.lcpSlvr_bbpgd.polyDisperseRatio_0.2verification.new.diary.log',
    'amphi.lattice.n_6.p_8.cDist_3.lcpSlvr_proxquasinewton.polyDisperseRatio_0.2verification.new.diary.log',
    'amphi.lattice.n_6.p_8.cDist_3.lcpSlvr_bifi.polyDisperseRatio_0.2verification.pLo_4.diary.log',
    'amphi.lattice.n_6.p_8.cDist_3.lcpSlvr_bifi.polyDisperseRatio_0.2_noWarmStart.diary.log'
]
retDic = {}
for algo, fn in zip(algo_names, file_names):
    print('==============================================')
    print(f'{algo} : {fn}')
    retDic[algo] = scrape_log(fn)

BB-PGD : amphi.lattice.n_6.p_8.cDist_3.lcpSlvr_bbpgd.polyDisperseRatio_0.2verification.new.diary.log
- n=216
- _mvp_times=[153.0, 151.0, 158.0, 146.0, 142.0, 135.0, 135.0, 134.0, 139.0, 135.0, 135.0, 135.0, 134.0, 140.0, 135.0, 135.0, 140.0]
- _lofi_mvp_times=[]
- lcp_solve_time=2381.52
- n=216
- _mvp_times=[131.0, 130.0, 126.0, 133.0, 125.0, 126.0, 131.0, 126.0, 126.0, 126.0, 133.0, 133.0, 131.0, 135.0, 135.0]
- _lofi_mvp_times=[]
- lcp_solve_time=1949.15
- n=216
- _mvp_times=[130.0, 131.0, 135.0, 131.0, 131.0, 131.0, 131.0, 131.0, 127.0, 127.0]
- _lofi_mvp_times=[]
- lcp_solve_time=1305.13
- n=216
- _mvp_times=[131.0, 131.0, 134.0, 131.0, 131.0, 131.0, 131.0, 130.0, 130.0, 127.0]
- _lofi_mvp_times=[]
- lcp_solve_time=1307.37
- n=216
- _mvp_times=[131.0, 131.0, 135.0, 131.0, 132.0, 134.0, 134.0, 131.0, 131.0, 127.0]
- _lofi_mvp_times=[]
- lcp_solve_time=1315.85
- n=216
- _mvp_times=[131.0, 132.0, 137.0, 133.0, 130.0, 131.0, 131.0, 132.0, 132.0, 127.0]
- _lofi_mvp_times=[]
- lcp_solve_

IndexError: list index out of range

In [27]:
retDic

{'BB-PGD': {'lcp_sizes': [216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
   216,
 